# R1-02b Synthetic Spreekorb email set

Why: the Tobi-Bueck tickets are SaaS/IT topics (0 of 50 held-out fully fit Spreekorb; this comes from Ayhan manual review, not a sourced figure), so the triage test, timing test and n8n demo use a synthetic Spreekorb set. Shopzilla stays for the dashboard.

Output: `data/processed/spreekorb_emails_60.csv`, `langsmith/heldout_labels_v2.csv`, a section in `data/README.md`. All emails are synthetic (made by gpt-4o-mini).

Order number rule: every topic needs an order number (SPK- plus 6 digits) in the body, except How-to / product question and Thanks / feedback, where it is optional (a real customer often has none). If present, the format must be right.

Cells: 1 Check, 2 Backup, 3 Create (3a grid + split, 3b generate, 3c labels file, 3d README), 4 Verify, 5 Unchanged, 6 Undo, 7 Git.

In [ ]:
# Cell 1 Check: right folder, input files exist (stops if a required file is missing)
import os, re, csv, json, random, shutil, hashlib
from datetime import datetime
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

RUN_UNDO = False      # True = restore backups and delete the new files (cell 6)
REGENERATE = False    # True = call OpenAI again even if the CSV already exists

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def check(name, ok):
    print('PASS' if ok else 'FAIL', '|', name)
    return bool(ok)

def warn(name, ok):
    print('PASS' if ok else 'WARN (manual review)', '|', name)
    return bool(ok)

PROC = REPO / 'data' / 'processed'
EMAILS = PROC / 'spreekorb_emails_60.csv'
LABELS = REPO / 'langsmith' / 'heldout_labels_v2.csv'
README = REPO / 'data' / 'README.md'
TRIAGE = REPO / 'n8n' / 'triage_prompt.md'
KB = REPO / 'knowledgebase' / 'kb_compiled.md'
# files this card must not change
UNCHANGED = [PROC / n for n in ['shopzilla_contacts_clean.csv', 'shopzilla_email_clean.csv', 'tobi_tuning_150.csv', 'tobi_test_50.csv']]
UNCHANGED.append(REPO / 'langsmith' / 'heldout_priority_labels.csv')

check('repo root found (AGENTS.md)', (REPO / 'AGENTS.md').exists())
missing = []
for p in [README, TRIAGE, KB] + UNCHANGED:
    ok = check('input exists: ' + str(p.relative_to(REPO)), p.exists())
    if not ok:
        missing.append(str(p.relative_to(REPO)))
if missing:
    raise FileNotFoundError('Required input files missing, stopping: ' + ', '.join(missing))

# Ayhan accepted this pattern (same as R1-05): load the key from .env into the environment, never print it
load_dotenv(REPO / '.env', override=False)
check('OPENAI_API_KEY is set (value not shown)', bool(os.environ.get('OPENAI_API_KEY')))

In [ ]:
# Cell 2 Backup: copy every file this card may change to _backup/<timestamp>_R1-02b/ and write a manifest
STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP = REPO / '_backup' / (STAMP + '_R1-02b')
BACKUP.mkdir(parents=True, exist_ok=True)
existed_before = {}
for p in [README, EMAILS, LABELS]:
    existed_before[p] = p.exists()
    if p.exists():
        dest = BACKUP / p.relative_to(REPO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
        print('backed up:', p.relative_to(REPO))
    else:
        print('new file, nothing to back up:', p.relative_to(REPO))
# manifest: which target files existed before this run (undo reads it)
manifest = {str(p.relative_to(REPO)): ex for p, ex in existed_before.items()}
(BACKUP / 'manifest.json').write_text(json.dumps(manifest, indent=1), encoding='utf-8')

# fingerprints of the files we do not own (compared in cell 5)
def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
hash_before = {p: sha(p) for p in UNCHANGED}
print('backup folder:', BACKUP.relative_to(REPO), '| hashes stored:', len(hash_before))

In [ ]:
# Cell 3a Create: grid and split. This runs BEFORE any AI output exists.
# Grid note: the grid was raised from 56 to 60 emails (+1 each for Order status, Invoice resend,
# Double charge and How-to / product question). Ayhan approved this on 27 Nov.
# (topic, queue_intended, priority_intended, count). Policy numbers left out of the topic text.
GRID = [
    ('Order status', 'Customer Service', 'medium', 5),
    ('Late delivery, no tracking update', 'Customer Service', 'medium', 5),
    ('Cancellation before dispatch', 'Customer Service', 'medium', 4),
    ('Return request', 'Returns and Exchanges', 'medium', 5),
    ('Exchange, wrong size', 'Returns and Exchanges', 'medium', 4),
    ('Wrong item received', 'Returns and Exchanges', 'medium', 4),
    ('Refund overdue', 'Billing and Payments', 'high', 5),
    ('Invoice resend', 'Billing and Payments', 'medium', 5),
    ('Double charge', 'Billing and Payments', 'high', 5),
    ('Damaged item', 'Product Support', 'high', 5),
    ('How-to / product question', 'Product Support', 'low', 5),
    ('Data protection deletion request', 'Customer Service', 'critical', 3),
    ('Legal threat', 'Customer Service', 'critical', 3),
    ('Thanks / feedback', 'Customer Service', 'very_low', 2),
]
OPTIONAL_ORDER = ['How-to / product question', 'Thanks / feedback']   # order number optional here
TONES = ['calm', 'annoyed', 'short-unclear', 'long-two-requests']
NAMES = ['Lena', 'Jonas', 'Mira', 'Tarek', 'Sofie', 'Henrik', 'Amara', 'Luca', 'Nadia', 'Pavel', 'Greta', 'Omar']  # invented
LANG_NAME = {'de': 'German', 'en': 'English'}

# expand to 60 rows in grid order: tone rotates, language alternates (30/30), order number from seed 42
num_rng = random.Random(42)
rows = []
for topic, q, pr, n in GRID:
    for _ in range(n):
        i = len(rows)
        rows.append({'email_id': 'S%02d' % (i + 1), 'topic': topic, 'tone': TONES[i % 4],
                     'language': 'de' if i % 2 == 0 else 'en',
                     'queue_intended': q, 'priority_intended': pr,
                     'name': NAMES[i % len(NAMES)], 'order_no': 'SPK-%06d' % num_rng.randint(0, 999999)})
grid_df = pd.DataFrame(rows)

# heldout size per topic: floor of count/3; a topic with 0 gets 1 so every topic is in heldout;
# the remaining slots go by largest remainder of count/3, ties broken by seed 42
srng = random.Random(42)
HELD_TOTAL = 20
quota = {t: n // 3 for t, _, _, n in GRID}
forced = [t for t in quota if quota[t] == 0]
for t in forced:
    quota[t] = 1
remainder = {t: (n % 3) / 3 for t, _, _, n in GRID if t not in forced}
left = HELD_TOTAL - sum(quota.values())
ranked = sorted(remainder, key=lambda t: (-remainder[t], srng.random()))
for t in ranked[:left]:
    quota[t] += 1
assert all(quota[t] <= n - 1 for t, _, _, n in GRID), 'every topic must keep 1 row in tuning'

# pick the heldout rows per topic; retry (seeded) until about 10 de / 10 en
for attempt in range(2000):
    held = []
    for t, _, _, n in GRID:
        ids = grid_df.index[grid_df['topic'] == t].tolist()
        held += srng.sample(ids, quota[t])
    if (grid_df.loc[held, 'language'] == 'de').sum() == 10:
        break

# 10 timing rows inside heldout: at least 4 de, 4 en, at least 7 different topics
for attempt in range(2000):
    timing = srng.sample(held, 10)
    lang = grid_df.loc[timing, 'language']
    if (lang == 'de').sum() >= 4 and (lang == 'en').sum() >= 4 and grid_df.loc[timing, 'topic'].nunique() >= 7:
        break

grid_df['split'] = 'tuning'
grid_df.loc[held, 'split'] = 'heldout'
grid_df['timing_flag'] = False
grid_df.loc[timing, 'timing_flag'] = True
print('split:', grid_df['split'].value_counts().to_dict(), '| timing:', int(grid_df['timing_flag'].sum()))
print('heldout languages:', grid_df[grid_df['split'] == 'heldout']['language'].value_counts().to_dict())

In [ ]:
# Cell 3b Create: generate the 60 emails with gpt-4o-mini (or load the CSV if it exists)
# Ayhan accepted: the generator prompt carries these extras (name, order number, length, no-invention rule),
# and the key is loaded with load_dotenv as in R1-05. No triage text, no knowledge base in the prompt.
GEN_PROMPT = '''You are a customer of Spreekorb, an online shop in Berlin.
Write one email from you to the Spreekorb customer service.
Topic: {topic}
Tone: {tone}
Language: {language}
Your first name: {name}
Order number: {order_line}
Length: the body has 40 to 150 words.
Rules: do not use em dashes. If the language is German, write in a natural customer voice, like a real person. Sign with your first name.
Do not invent a surname, email address, phone number, street address, IBAN, company, law firm or court name.
Return JSON with the keys subject and body.'''

TONE_TEXT = {'calm': 'calm and polite',
             'annoyed': 'annoyed, but not abusive',
             'short-unclear': 'short and a bit unclear, few details (still at least 40 words)',
             'long-two-requests': 'long, with two separate requests in one email'}
EM = chr(8212)   # em dash character

# patterns for personal-data-like text (used by the retry here and by Verify)
ADDR = r'[0-9]{5} +[A-ZÄÖÜ][a-zäöüß]+|[A-ZÄÖÜ][a-zäöüß]*(?:straße|strasse|weg|allee|platz|str[.]) *[0-9]+'
LEGAL = r'(?i:Kanzlei|Rechtsanwalt|law firm|attorney at|court of|Amtsgericht|Landgericht)[ ]+[A-ZÄÖÜ][a-zäöüß]+'

def pii_fail(text):
    # returns the list of hard problems found in a text (SPK order numbers are ignored)
    t = re.sub(r'SPK-[0-9]{6}', ' ', text)
    out = []
    if '@' in t:
        out.append('@')
    if any(sum(c.isdigit() for c in m) >= 7 for m in re.findall(r'[0-9][0-9 ()/+-]*[0-9]', t)):
        out.append('phone-like digits')
    if re.search(r'DE[0-9]{20}', t.replace(' ', '')):
        out.append('IBAN')
    if re.search(ADDR, t):
        out.append('postcode/street')
    return out

def build_prompt(r):
    if r['topic'] in OPTIONAL_ORDER:
        order_line = 'optional, you may mention ' + r['order_no'] + ' or leave it out'
    else:
        order_line = 'write this exact order number in the body: ' + r['order_no']
    return GEN_PROMPT.format(topic=r['topic'], tone=TONE_TEXT[r['tone']], language=LANG_NAME[r['language']],
                             name=r['name'], order_line=order_line)

def problem(subj, body, r):
    # returns '' if the email is fine, else the reason (used for the retry)
    words = len(body.split())
    if not subj.strip() or not body.strip():
        return 'empty'
    if words < 40 or words > 150:
        return 'words=%d' % words
    if EM in subj or EM in body:
        return 'em dash'
    if r['topic'] not in OPTIONAL_ORDER and r['order_no'] not in body:
        return 'order number missing'
    if any(t != r['order_no'] for t in re.findall(r'SPK-[0-9A-Za-z]*', subj + ' ' + body)):
        return 'wrong order number'
    if pii_fail(subj + ' ' + body):
        return 'personal-data-like text'
    return ''

PRICE_IN_PER_M = None    # USD per 1M input tokens, fill from the OpenAI pricing page (not invented here)
PRICE_OUT_PER_M = None   # USD per 1M output tokens

if EMAILS.exists() and not REGENERATE:
    print('LOCK: CSV exists, skipping generation (set REGENERATE = True to redo)')
    emails = pd.read_csv(EMAILS, dtype=str).fillna('')
else:
    from openai import OpenAI
    oai = OpenAI()    # reads OPENAI_API_KEY from the environment
    tok_in = tok_out = 0
    subjects, bodies = [], []
    for k, r in grid_df.iterrows():
        subj = body = ''
        for attempt in range(1, 4):    # max 3 tries
            resp = oai.chat.completions.create(
                model='gpt-4o-mini', temperature=0.9, response_format={'type': 'json_object'},
                messages=[{'role': 'user', 'content': build_prompt(r)}])
            tok_in += resp.usage.prompt_tokens
            tok_out += resp.usage.completion_tokens
            try:
                data = json.loads(resp.choices[0].message.content)
                subj, body = str(data.get('subject', '')), str(data.get('body', ''))
            except Exception:
                subj = body = ''
            why = problem(subj, body, r)
            if not why:
                break
            print('   retry', attempt, r['email_id'], why)
        subjects.append(subj)
        bodies.append(body)
        print('%s %d/60 %s %s %s' % (r['email_id'], k + 1, r['language'], r['tone'], 'ok' if not why else 'WARN ' + why))
    emails = grid_df.copy()
    emails['subject'], emails['body'] = subjects, bodies
    emails['synthetic'] = True
    emails['generator'] = 'gpt-4o-mini, temp 0.9, R1-02b'
    cols = ['email_id', 'topic', 'tone', 'language', 'subject', 'body', 'queue_intended', 'priority_intended',
            'split', 'timing_flag', 'synthetic', 'generator']
    emails[cols].to_csv(EMAILS, index=False)
    print('saved', EMAILS.relative_to(REPO))
    print('tokens (from API usage): in', tok_in, 'out', tok_out)
    if PRICE_IN_PER_M is not None and PRICE_OUT_PER_M is not None:
        print('cost estimate USD:', round(tok_in / 1e6 * PRICE_IN_PER_M + tok_out / 1e6 * PRICE_OUT_PER_M, 4), '(estimate)')
    else:
        print('cost: set PRICE_IN_PER_M and PRICE_OUT_PER_M to get an estimate')

In [ ]:
# Cell 3c Create: heldout labels file for Ayhan (no AI triage outputs, no intended queue/priority: avoids anchoring)
LABEL_COLS = ['email_id', 'topic', 'language', 'subject', 'body', 'queue_human', 'priority_human', 'note']
src = pd.read_csv(EMAILS, dtype=str).fillna('')
held_df = src[src['split'] == 'heldout'].copy()
for c in ['queue_human', 'priority_human', 'note']:
    held_df[c] = ''

# keep the old sheet only if: same ids, same subject AND body as the current CSV, and a human label is filled
# (the old file was already backed up in cell 2)
labels_kept = False
if LABELS.exists():
    old = pd.read_csv(LABELS, dtype=str).fillna('')
    if set(old['email_id']) == set(held_df['email_id']) and len(old) == len(held_df):
        cur = held_df.set_index('email_id')
        same_text = all(cur.loc[e, 'subject'] == s and cur.loc[e, 'body'] == b
                        for e, s, b in zip(old['email_id'], old['subject'], old['body']))
        filled = old['queue_human'].str.strip().ne('').any() or old['priority_human'].str.strip().ne('').any()
        labels_kept = bool(same_text and filled)
if labels_kept:
    # keep the human labels, but drop old columns such as queue_intended
    old.reindex(columns=LABEL_COLS).fillna('').to_csv(LABELS, index=False)
    print('KEEP: human labels kept for the same emails (columns tidied)')
else:
    held_df[LABEL_COLS].to_csv(LABELS, index=False)
    print('saved', LABELS.relative_to(REPO), '| rows', len(held_df))

In [ ]:
# Cell 3d Create: README section between markers (replaced if it exists, so this cell is safe to re-run)
NL = chr(10)
START, END = '<!-- R1-02b start -->', '<!-- R1-02b end -->'
section = NL.join([
    START,
    '## R1-02b Synthetic Spreekorb email set',
    '',
    'Why: the Tobi-Bueck tickets proved to be SaaS/IT topics even after the shop-keyword filter (0 of 50 held-out fully fit Spreekorb; this comes from Ayhan manual review, not a sourced figure). So the triage test, the timing test and the n8n demo use a synthetic Spreekorb set. Shopzilla stays for the dashboard.',
    '',
    '- `data/processed/spreekorb_emails_60.csv`: 60 synthetic emails (S01-S60), 30 de / 30 en, 14 topics, written by gpt-4o-mini (temperature 0.9). Column synthetic = True.',
    '- All emails are AI-generated synthetic test data (gpt-4o-mini), not real customers, and not for production or legal use.',
    '- Split made in code from the grid before any AI output: 40 tuning, 20 heldout, 10 heldout rows have timing_flag = True.',
    '- The grid was raised from 56 to 60 (+1 to Order status, Invoice resend, Double charge and How-to / product question; Ayhan approved, 27 Nov).',
    '- Order number SPK- plus 6 digits is set in Python (seed 42). It is required in the body, except for How-to and Thanks emails (optional).',
    '- queue_intended and priority_intended are the generator intentions, not ground truth. The queue_human and priority_human labels from Ayhan are the official truth.',
    '- `langsmith/heldout_labels_v2.csv`: the 20 heldout emails with empty queue_human / priority_human for Ayhan to fill (the intended values are left out on purpose, to avoid anchoring).',
    '- Triage output is a draft, and a human approves every reply (human oversight).',
    '- The deletion-request and legal-threat emails are synthetic routing tests for escalation to a human or legal.',
    '- Results are reported at whole-set or queue level only, since there are 1-2 heldout rows per topic.',
    '- Notebook: `notebooks/R1-02b_synthetic_emails.ipynb`. Note: re-running R1-02 rewrites this README and drops this section; re-run R1-02b cell 3d to add it back.',
    END])
text = README.read_text(encoding='utf-8')
if START in text and END in text:
    a, b = text.index(START), text.index(END) + len(END)
    text = text[:a] + section + text[b:]
else:
    text = text.rstrip() + NL + NL + section + NL
README.write_text(text, encoding='utf-8')
print('README section written, blocks:', text.count(START))

In [ ]:
# Cell 4 Verify: every acceptance item prints PASS/FAIL
df = pd.read_csv(EMAILS, dtype=str).fillna('')
exp = {t: (q, p, n) for t, q, p, n in GRID}
words = df['body'].str.split().str.len()
both = df['subject'] + ' ' + df['body']

check('60 rows', len(df) == 60)
check('30 de / 30 en', (df['language'] == 'de').sum() == 30 and (df['language'] == 'en').sum() == 30)
check('counts per topic match the grid', all((df['topic'] == t).sum() == v[2] for t, v in exp.items()) and set(df['topic']) == set(exp))
check('queue and priority per topic match the grid',
      all(((df['topic'] == t) & (df['queue_intended'] == v[0]) & (df['priority_intended'] == v[1])).sum() == v[2] for t, v in exp.items()))
check('split is 40 tuning / 20 heldout', (df['split'] == 'tuning').sum() == 40 and (df['split'] == 'heldout').sum() == 20)
tim = df['timing_flag'] == 'True'
check('10 timing flags, all in heldout', tim.sum() == 10 and (df[tim]['split'] == 'heldout').all())
check('timing flags: at least 4 de and 4 en', (df[tim]['language'] == 'de').sum() >= 4 and (df[tim]['language'] == 'en').sum() >= 4)
print('INFO | heldout languages:', df[df['split'] == 'heldout']['language'].value_counts().to_dict(), '| timing topics:', df[tim]['topic'].nunique())
check('split in the CSV equals the split made in code before the AI', df['split'].tolist() == grid_df['split'].tolist() and tim.tolist() == grid_df['timing_flag'].tolist())
check('every topic appears in tuning', set(df[df['split'] == 'tuning']['topic']) == set(exp))
check('every topic appears in heldout', set(df[df['split'] == 'heldout']['topic']) == set(exp))
print('INFO | heldout per topic:', df[df['split'] == 'heldout']['topic'].value_counts().to_dict())
check('ids are S01-S60 and unique', df['email_id'].tolist() == ['S%02d' % i for i in range(1, 61)])
check('no empty subject or body', (df['subject'].str.strip() != '').all() and (df['body'].str.strip() != '').all())
check('body word count 40-150', ((words >= 40) & (words <= 150)).all())

# order numbers: any SPK- token must be SPK- plus 6 digits; required topics must have one
tokens = both.apply(lambda s: re.findall(r'SPK-[0-9A-Za-z]*', s))
check('order numbers match SPK-6 digits wherever present', all(re.fullmatch(r'SPK-[0-9]{6}', t) for ts in tokens for t in ts))
need = ~df['topic'].isin(OPTIONAL_ORDER)
check('every topic that requires an order number has one', (tokens[need].str.len() > 0).all())
check('no em dash in subject or body', not df['subject'].str.contains(EM).any() and not df['body'].str.contains(EM).any())
check('synthetic is True everywhere', (df['synthetic'] == 'True').all())

# no invented personal data (subject + body)
hits = both.apply(pii_fail)
check('no @ (email address)', not hits.apply(lambda h: '@' in h).any())
check('no phone-like digit runs (7+ digits, SPK number ignored)', not hits.apply(lambda h: 'phone-like digits' in h).any())
check('no IBAN pattern (DE + 20 digits)', not hits.apply(lambda h: 'IBAN' in h).any())
check('no German postcode + street pattern', not hits.apply(lambda h: 'postcode/street' in h).any())
legal_hits = both[both.str.contains(LEGAL, regex=True)]
warn('no law firm / court followed by a proper name', len(legal_hits) == 0)
for i in legal_hits.index:
    print('   review', df.loc[i, 'email_id'], re.search(LEGAL, legal_hits[i]).group(0))

# the generator prompt must not contain triage prompt text or knowledge base text
def norm_words(s):
    return re.findall(r'[^ \t\r\n]+', s.lower())
def windows(s, n=8):
    w = norm_words(s)
    return set(' '.join(w[i:i + n]) for i in range(len(w) - n + 1))
PROMPT_WIN = windows(GEN_PROMPT)
def leaked_lines(path):
    lines = [l.strip() for l in path.read_text(encoding='utf-8').splitlines()]
    return [l for l in lines if len(l) > 30 and l in GEN_PROMPT]
def leaked_windows(path):
    return PROMPT_WIN & windows(path.read_text(encoding='utf-8'))
check('GEN_PROMPT has no triage prompt text (lines and 8-word windows)', leaked_lines(TRIAGE) == [] and not leaked_windows(TRIAGE))
check('GEN_PROMPT has no knowledge base text (lines and 8-word windows)', leaked_lines(KB) == [] and not leaked_windows(KB))

# heldout labels file
lab = pd.read_csv(LABELS, dtype=str).fillna('')
held_ids = df[df['split'] == 'heldout']['email_id'].tolist()
check('labels file has exactly the right columns (no intended queue/priority)', list(lab.columns) == LABEL_COLS)
check('labels file has 20 rows and the heldout ids', len(lab) == 20 and sorted(lab['email_id']) == sorted(held_ids))
blank = (lab['queue_human'].str.strip() == '').all() and (lab['priority_human'].str.strip() == '').all()
check('human columns blank or kept', blank or labels_kept)

rd = README.read_text(encoding='utf-8')
check('README has the R1-02b section with the reason', START in rd and END in rd and '0 of 50' in rd and 'manual review' in rd)

In [ ]:
# Cell 5 Unchanged: Shopzilla, Tobi and the old heldout labels must be byte-identical
for p in UNCHANGED:
    check('unchanged: ' + str(p.relative_to(REPO)), sha(p) == hash_before[p])

In [ ]:
# Cell 6 Undo: only runs if RUN_UNDO = True
# Uses the OLDEST R1-02b backup folder (the state before the first run), so undo is safe after re-runs.
if RUN_UNDO:
    folders = sorted((REPO / '_backup').glob('*_R1-02b'))
    if not folders:
        print('FAIL | no R1-02b backup folder found')
    else:
        oldest = folders[0]
        print('using backup folder:', oldest.relative_to(REPO))
        old_manifest = json.loads((oldest / 'manifest.json').read_text(encoding='utf-8'))
        for rel, existed in old_manifest.items():
            p = REPO / rel
            if existed:
                shutil.copy2(oldest / rel, p)    # restore the file that existed before
                print('restored:', rel)
            elif p.exists():
                p.unlink()                        # delete a file this card created
                print('deleted:', rel)
else:
    print('RUN_UNDO is False, nothing restored')

In [ ]:
# Cell 7 Git: print the commands only, Ayhan runs them
print('git status')
print('git add notebooks/R1-02b_synthetic_emails.ipynb data/processed/spreekorb_emails_60.csv langsmith/heldout_labels_v2.csv data/README.md')
print('git commit -m "R1-02b synthetic Spreekorb email set"')